# Exemplo de Execução do Pipeline (CNN 2D)

Este notebook demonstra o uso passo a passo dos componentes do pipeline de Machine Learning.

In [2]:
import os
import sys
import torch
import numpy as np
from sklearn.model_selection import train_test_split

# Em notebooks, __file__ não existe por padrão. Usamos o diretório atual para encontrar a raiz do projeto.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from ai.loader.loader import DataLoader
from ai.label.label_generator import LabelGenerator
from ai.preprocess.cnn2d import PreprocessCNN2D
from ai.models.cnn2d import ModelCNN2D
from ai.trainer.trainer import ModelTrainer
from ai.evaluation.monitor import ModelMonitor
from ai.evaluation.summary import ModelSummary
from ai.preprocess.balancer import DataBalancer

## 1. Carregamento dos Dados

Vamos usar a classe `DataLoader` para buscar os arquivos parquet e carregá-los em um DataFrame do Pandas.

In [3]:
# Ajuste o data_path para apontar para seus arquivos reais de dados.
# Se não passar nada, o loader buscará por padrão na pasta data/parquet/
data_path = os.path.join(PROJECT_ROOT, "data", "parquet", "**", "*.parquet")

loader = DataLoader(data_path=data_path, max_files=2) # Limitando a 1 arquivo para o exemplo rodar rápido
df = loader.execute()

if df is None or df.empty:
    print("Erro: Nenhum dado foi carregado.")

print("\n-> Geração de Labels...")
df = LabelGenerator.apply_label(df, file_path_col='file_path', label_col="label")
print("DataFrame\n")
print("-----CAMINHO-----")
print(df["file_path"][100])
print("-----LABEL-----")
print(df["label"][100])
print("-----CAMINHO 2-----")
print(df["file_path"][7500])
print("-----LABEL-----")
print(df["label"][7500])

Encontrados 8 arquivos válidos


Carregando Parquets: 100%|██████████| 8/8 [00:00<00:00,  9.13arquivo/s]


-> Geração de Labels...
DataFrame

-----CAMINHO-----
/home/joao.gomes/LPS/cern/data/parquet/mc25_13TeV.20260104.physics_Main.JF17.100k.r1.JF17.parquet/JF17.0.parquet
-----LABEL-----
0
-----CAMINHO 2-----
/home/joao.gomes/LPS/cern/data/parquet/mc25_13TeV.20251215.physics_Main.Zee.500k.r1.Zee.parquet/Zee.1.parquet
-----LABEL-----
1


## 2. Pré-processamento

O pré-processador formata as variáveis em tensores adequados para a CNN (imagens 2D com múltiplos canais).

In [4]:
preprocessor = PreprocessCNN2D()

X = preprocessor.transform(df)
Y = preprocessor.get_labels(df, label_col='label')
    
print(f"Formato de X (Features): {X.shape}")
print(f"Formato de Y (Labels): {Y.shape}")

print("Entrada:",X)
print("Labels:",Y)


Convertendo camadas do calorímetro em Imagens 2D (Tensores)...
[1/7] Processando canal: cl_cells_presampler


Processando Amostras: 100%|██████████| 15100/15100 [00:00<00:00, 23241.26it/s]


[2/7] Processando canal: cl_cells_em1


Processando Amostras: 100%|██████████| 15100/15100 [00:00<00:00, 24924.66it/s]


[3/7] Processando canal: cl_cells_em2


...

[6/7] Processando canal: cl_cells_had2


Processando Amostras: 100%|██████████| 15100/15100 [00:00<00:00, 21468.25it/s]


[7/7] Processando canal: cl_cells_had3


Processando Amostras: 100%|██████████| 15100/15100 [00:00<00:00, 22375.91it/s]

Formato de X (Features): (15100, 7, 7, 15)
Formato de Y (Labels): (15100,)
Entrada: [[[[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  ...

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   ...
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]]

  [[0. 0. 0. ... 0. 0. 0.]
   [0. 0. 0. ... 0. 0. 0.]


## 3. Balanceamento dos Dados

Realiza o balanceamento dos dados utilizando Undersampling

In [8]:
data_balancer = DataBalancer()
X_balanced, Y_balanced = data_balancer.apply(X, Y)

DataBalancer: Balanceando para 641 amostras por classe (Undersampling)...


## 4. Divisão de Dados (Train / Test)

Separamos um conjunto de teste isolado.

In [10]:
X_train, X_test, Y_train, Y_test = train_test_split(X_balanced, Y_balanced, test_size=0.15, random_state=42, shuffle = True)
print(f"Treinamento: {X_train.shape[0]} amostras")
print(f"Teste Isolado: {X_test.shape[0]} amostras")
print(Y_train)
print(Y_test)

uns_train = np.sum(Y_train == 1.0)
zeros_train = np.sum(Y_train == 0.0)

uns_test = np.sum(Y_test == 1.0)
zeros_test = np.sum(Y_test == 0.0)

print("Quantidade de 1 no conjunto de teste:",uns_test)
print("Quantidade de 1 no conjunto de treino:",uns_train)
print("Quantidade de 0 no conjunto de teste:",zeros_test)
print("Quantidade de 0 no conjunto de treino:",zeros_train)

Treinamento: 1089 amostras
Teste Isolado: 193 amostras
[1. 1. 0. ... 1. 1. 0.]
[1. 0. 0. 0. 1. 0. 0. 0. 1. 1. 0. 0. 1. 1. 1. 1. 1. 1. 0. 0. 1. 0. 0. 1.
 1. 1. 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 1. 1. 1. 0. 0. 0. 0. 1. 1. 0. 0.
 1. 1. 1. 1. 0. 0. 0. 1. 0. 1. 1. 1. 1. 1. 1. 1. 0. 1. 0. 1. 0. 0. 1. 1.
 1. 0. 0. 0. 0. 1. 1. 0. 1. 1. 1. 0. 1. 1. 0. 1. 1. 1. 1. 0. 0. 1. 1. 1.
 0. 0. 1. 1. 1. 0. 0. 1. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 1. 1. 1. 1. 0.
 1. 0. 0. 1. 1. 1. 0. 0. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 1. 1. 1. 0.
 1. 1. 1. 0. 1. 1. 0. 0. 0. 1. 0. 0. 0. 1. 0. 1. 0. 1. 0. 1. 1. 1. 0. 1.
 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 0. 0. 1. 0. 0. 1. 0. 1. 0. 0. 0.
 1.]
Quantidade de 1 no conjunto de teste: 103
Quantidade de 1 no conjunto de treino: 538
Quantidade de 0 no conjunto de teste: 90
Quantidade de 0 no conjunto de treino: 551


## 5. Treinamento

Configuramos o treinador (`ModelTrainer`) que lidará com o ciclo de vida do PyTorch Lightning. Podemos usar Holdout Simples ou K-Fold.

In [11]:
results_dir = os.path.join(PROJECT_ROOT, "results", "CNN2D")

trainer = ModelTrainer(
    max_epochs=2,
    batch_size=32,
    patience=3,
    log_dir=os.path.join(results_dir, "lightning_logs")
)

model_kwargs = {'learning_rate': 0.001}

fold_trainers, fold_models = trainer.fit_kfold(
    ModelCNN2D, model_kwargs, X_train, Y_train, 
    n_splits=3, target_fold=None
)

    
# Treino Simples (Holdout)
print("\nIniciando treinamento simples...")
model = ModelCNN2D(learning_rate=0.001)
trained_trainer = trainer.fit(model, X_train, Y_train)

/home/joao.gomes/LPS/cern/neuralnet-env/lib/python3.13/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/joao.gomes/LPS/cern/neuralnet-env/lib/python3. ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/joao.gomes/LPS/cern/neuralnet-env/lib/python3.13/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/joao.gomes/LPS/cern/results/CNN2D/lightning_logs/fold_1 exists and is not empty.

  | Name       | Type              | Params | Mode  | FLOPs
-----------------------------------------------

Iniciando Validação Cruzada com 3 folds...

==================== Fold 1/3 ====================
                                                                            

/home/joao.gomes/LPS/cern/neuralnet-env/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/joao.gomes/LPS/cern/neuralnet-env/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/home/joao.gomes/LPS/cern/neuralnet-env/lib/python3.13/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/home/joao.gomes/LPS/cern/neuralnet-env/lib/python3.13/site-packages/pytorch_lightning/loops/fit_loop.py:321: The

Epoch 0: 100%|██████████| 23/23 [00:00<00:00, 88.58it/s, v_num=1, train_loss_step=0.717, val_loss=0.624, val_acc=0.645, val_auc=0.807, train_loss_epoch=0.674, train_acc=0.565]

Metric val_loss improved. New best score: 0.624


...

Epoch 0: 100%|██████████| 28/28 [00:00<00:00, 106.99it/s, v_num=1, train_loss_step=0.590, val_loss=0.636, val_acc=0.756, val_auc=0.807, train_loss_epoch=0.692, train_acc=0.517]

Metric val_loss improved. New best score: 0.636


Epoch 1: 100%|██████████| 28/28 [00:00<00:00, 74.04it/s, v_num=1, train_loss_step=0.519, val_loss=0.515, val_acc=0.765, val_auc=0.827, train_loss_epoch=0.563, train_acc=0.712] 

Metric val_loss improved by 0.121 >= min_delta = 0.0. New best score: 0.515
`Trainer.fit` stopped: `max_epochs=2` reached.


Epoch 1: 100%|██████████| 28/28 [00:00<00:00, 61.21it/s, v_num=1, train_loss_step=0.519, val_loss=0.515, val_acc=0.765, val_auc=0.827, train_loss_epoch=0.563, train_acc=0.712]
Treinamento finalizado! Melhor modelo: /home/joao.gomes/LPS/cern/results/CNN2D/lightning_logs/ModelCNN2D-epoch=01-val_loss=0.5149.ckpt


## 6. Avaliação

Vamos avaliar o modelo treinado em nosso Test Set (dados isolados).

In [ ]:
results_dir=os.path.join("results", "CNN_2D_Test")
summary = ModelSummary(output_dir=os.path.join(results_dir, "metrics"))
monitor = ModelMonitor(output_dir=os.path.join(results_dir, "plots"))
model.eval()
with torch.no_grad():
    X_tensor = torch.as_tensor(X_test, dtype=torch.float32)
    logits = model(X_tensor)
    y_prob = torch.sigmoid(logits).numpy().flatten()

y_true = Y_test.flatten()
y_pred = (y_prob >= 0.8).astype(int)
        
file_suffix = f"fold_{fold_idx}"
summary.save_metrics(y_true, y_prob, threshold=0.8, filename=f"test_metrics{file_suffix}.csv")
monitor.plot_roc_curve(y_true, y_prob, filename=f"roc_curve{file_suffix}.pdf")
monitor.plot_confusion_matrix(y_true, y_pred, filename=f"confusion_matrix{file_suffix}.pdf")